In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/sample_submission.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test_labels.csv.zip
/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test.csv.zip


# Chose option train

In [2]:
train_on_RNNVanilla = False
train_on_RNNLSTM = False
train_on_Transformer = True

In [3]:
import torch
import torch.nn as nn
loss_function = nn.BCEWithLogitsLoss()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Đang huấn luyện trên: {device}")

Đang huấn luyện trên: cuda


In [4]:
import pandas as pd

# Đường dẫn đến các file trong Kaggle input
train_path = '/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/train.csv.zip'
test_path = '/kaggle/input/competitions/jigsaw-toxic-comment-classification-challenge/test.csv.zip'

# Đọc dữ liệu (Pandas tự xử lý nén zip)
df_train = pd.read_csv(train_path)
df_test = pd.read_csv(test_path)

# Xem qua cấu trúc
print(f"Train shape: {df_train.shape}")
print(df_train.head())

Train shape: (159571, 8)
                 id                                       comment_text  toxic  \
0  0000997932d777bf  Explanation\nWhy the edits made under my usern...      0   
1  000103f0d9cfb60f  D'aww! He matches this background colour I'm s...      0   
2  000113f07ec002fd  Hey man, I'm really not trying to edit war. It...      0   
3  0001b41b1c6bb37e  "\nMore\nI can't make any real suggestions on ...      0   
4  0001d958c54c6e35  You, sir, are my hero. Any chance you remember...      0   

   severe_toxic  obscene  threat  insult  identity_hate  
0             0        0       0       0              0  
1             0        0       0       0              0  
2             0        0       0       0              0  
3             0        0       0       0              0  
4             0        0       0       0              0  


In [5]:
# Tính số lượng từ của mỗi comment
train_lens = df_train['comment_text'].apply(lambda x: len(x.split()))
test_lens = df_test['comment_text'].apply(lambda x: len(x.split()))

print(f"Độ dài trung bình: {train_lens.mean():.2f}")
print(f"Độ dài lớn nhất: {train_lens.max()}")
print(f"95% dữ liệu có độ dài dưới: {train_lens.quantile(0.95)}")

Độ dài trung bình: 67.27
Độ dài lớn nhất: 1411
95% dữ liệu có độ dài dưới: 230.0


In [6]:
classes = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
targets = df_train[classes].values

# Tính tổng số lượng mẫu cho mỗi class
label_counts = df_train[classes].sum()
print("\nSố lượng mẫu của từng loại độc hại:")
print(label_counts)


Số lượng mẫu của từng loại độc hại:
toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64


In [7]:
import re
def simple_clean(text):
    text = text.lower()
    text = re.sub(r'[\n\t\r]', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df_train['comment_text'] = df_train['comment_text'].astype(str).apply(simple_clean)

df_train


,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,explanation why the edits made under my userna...,0,0,0,0,0,0
1,000103f0d9cfb60f,daww he matches this background colour im seem...,0,0,0,0,0,0
2,000113f07ec002fd,hey man im really not trying to edit war its j...,0,0,0,0,0,0
3,0001b41b1c6bb37e,more i cant make any real suggestions on impr...,0,0,0,0,0,0
4,0001d958c54c6e35,you sir are my hero any chance you remember wh...,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...
159566,ffe987279560d7ff,and for the second time of asking when your vi...,0,0,0,0,0,0
159567,ffea4adeee384e90,you should be ashamed of yourself that is a ...,0,0,0,0,0,0
159568,ffee36eab5c267c9,spitzer umm theres no actual article for pro...,0,0,0,0,0,0
159569,fff125370e4aaaf3,and it looks like it was actually you who put ...,0,0,0,0,0,0


In [8]:
from collections import Counter 
all_text = ' '.join(df_train['comment_text'].values)
words = all_text.split()
word_counts = Counter(words)

print("10 từ xuất hiện nhiều nhất:")
print(word_counts.most_common(10))

10 từ xuất hiện nhiều nhất:
[('the', 495480), ('to', 296851), ('of', 224024), ('and', 222368), ('a', 214910), ('you', 204560), ('i', 200655), ('is', 175960), ('that', 154301), ('in', 144188)]


In [9]:
classes = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]
targets = df_train[classes].values

# Tính tổng số lượng mẫu cho mỗi class
label_counts = df_train[classes].sum()
print("\nSố lượng mẫu của từng loại độc hại:")
print(label_counts)


Số lượng mẫu của từng loại độc hại:
toxic            15294
severe_toxic      1595
obscene           8449
threat             478
insult            7877
identity_hate     1405
dtype: int64


In [10]:
from sklearn.model_selection import train_test_split

# Assume X is your cleaned comments and y is your 6-column labels
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df_train['comment_text'].values, 
    df_train[['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate']].values, 
    test_size=0.1, 
    random_state=42
)



# Tokenize the data

In [11]:
from collections import Counter

def build_vocab(texts, max_vocab_size=10000):
    # 1. Count all words in the training set
    word_counts = Counter()
    for text in texts:
        word_counts.update(text.split())
    
    # 2. Keep only the most frequent words
    common_words = word_counts.most_common(max_vocab_size - 2)
    
    # 3. Create the mapping
    word_to_idx = {"<PAD>": 0, "<UNK>": 1}
    for i, (word, _) in enumerate(common_words):
        word_to_idx[word] = i + 2
        
    return word_to_idx

# 1. Xây dựng từ điển từ tập train
vocab = build_vocab(train_texts, max_vocab_size=30000)

In [12]:
import torch
from torch.utils.data import Dataset

class ToxicDataset(Dataset):
    def __init__(self, texts, labels, word_to_idx, max_len):
        self.texts = texts
        self.labels = labels
        self.word_to_idx = word_to_idx
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        # 1. Get text and labels for this index
        text = str(self.texts[idx]).split()
        label = self.labels[idx]

        # 2. Convert words to IDs (Use 1 for <UNK> if word not found)
        ids = [self.word_to_idx.get(word, 1) for word in text]

        # 3. Padding / Truncating
        if len(ids) < self.max_len:
            ids += [0] * (self.max_len - len(ids)) # Add <PAD> tokens
        else:
            ids = ids[:self.max_len] # Truncate

        return torch.tensor(ids), torch.tensor(label, dtype=torch.float32)

In [13]:
from torch.utils.data import DataLoader

# Create the Dataset objects
train_dataset = ToxicDataset(train_texts, train_labels, vocab, max_len=100)
val_dataset = ToxicDataset(val_texts, val_labels, vocab, max_len=100)

# Create the DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# Model RNN Vanilla

In [14]:
import torch
import torch.nn as nn

class VanillaRNNModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, device = 'cuda'):
        super(VanillaRNNModel, self).__init__()
        self.device = device

        self.hidden_dim = hidden_dim

        # Encoding every word into a vector of size embedding_dim
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # Processes the word embedding (x_t -> h_t)
        self.input_to_hidden = nn.Linear(embedding_dim, hidden_dim)

        # Processes the previous hidden state (the "memory") (h_(t-1) to h_t)
        self.hidden_to_hidden = nn.Linear(hidden_dim, hidden_dim)

        self.head_classifier = nn.Sequential(
            nn.Dropout(0.3),                               # 1. Tránh Overfitting
            nn.Linear(hidden_dim, hidden_dim // 2),        # 2. Giảm chiều dữ liệu từ từ (VD: 256 -> 128)
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),                                     # 3. Hàm kích hoạt phi tuyến tính
            nn.Dropout(0.3),                               # 4. Tránh Overfitting thêm lần nữa
            nn.Linear(hidden_dim // 2, output_dim)         # 5. Output ra 6 nhãn
        )
        
    def forward(self, x):
        # x shape: [Batch, Sequence Length]

        batch_size = x.size(0)
        sequence_length = x.size(1)

        # Initialize the first hidden state (h_0) with all zeros.
        # Ensure it's on the same device (CPU/GPU) as the input data.
        h_t = torch.zeros(batch_size, self.hidden_dim, device = self.device)
        
        # Shape becomes: [batch_size, seq_length, embedding_dim]
        embedded = self.embedding(x)

        for t in range(sequence_length):
            x_t = embedded[:, t, :]

            h_t = self.input_to_hidden(x_t) + self.hidden_to_hidden(h_t)
            h_t = torch.tanh(h_t)

        out = self.head_classifier(h_t)
        return out

    
        

In [15]:
class PytorchRNN(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, device = 'cuda'):
        super(PytorchRNN, self).__init__()
        self.device = device

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        self.rnn = nn.RNN(input_size = embedding_dim, hidden_size = hidden_dim, batch_first = True)
        
        self.head_classifier = nn.Sequential(
            nn.Dropout(0.3),                               # 1. Tránh Overfitting
            nn.Linear(hidden_dim, hidden_dim // 2),        # 2. Giảm chiều dữ liệu từ từ (VD: 256 -> 128)
            
            nn.ReLU(),                                     # 3. Hàm kích hoạt phi tuyến tính
            nn.Dropout(0.3),                               # 4. Tránh Overfitting thêm lần nữa
            nn.Linear(hidden_dim // 2, output_dim)         # 5. Output ra 6 nhãn
        ) 
    
    def forward(self, x):
        # x shape: [batch_size, sequence_length]
        embedded = self.embedding(x)

        # rnn_out: [batch, seq_len, hidden_dim] -> chứa toàn bộ các bước thời gian
        # hidden: [1, batch, hidden_dim] -> chỉ chứa bước cuối cùng
        rnn_out, hidden = self.rnn(embedded)

        # 1. Loại bỏ chiều thừa: [1, batch, hidden_dim] -> [batch, hidden_dim]
        last_hidden = hidden.squeeze(0)
        
        # 2. Truyền last_hidden vào classifier
        out = self.head_classifier(last_hidden) 
        
        return out

In [16]:
import time

if train_on_RNNVanilla:
    # 1. Khởi tạo
    model = VanillaRNNModel(vocab_size=len(vocab), embedding_dim=128, 
                            hidden_dim=256, output_dim=6, device=device).to(device)
    criterion = nn.BCEWithLogitsLoss() 
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    num_epochs = 60 # Bạn có thể để số epoch lớn hơn vì đã có Early Stopping
    best_val_loss = float('inf')
    
    # --- CẤU HÌNH PATIENCE ---
    patience = 5      # Nếu sau 3 Epoch mà Val Loss không giảm thì dừng
    counter = 0       # Biến đếm số lần không cải thiện
    early_stop = False

    print("Starting the training process with Early Stopping...")

    for epoch in range(num_epochs):
        if early_stop:
            print("🛑 Early stopping triggered. Training finished!")
            break

        start_time = time.time()
        
        # --- TRAINING PHASE ---
        model.train()
        train_loss = 0.0
        for texts, labels in train_loader:
            texts, labels = texts.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(texts)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()

        # --- VALIDATION PHASE ---
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for texts, labels in val_loader:
                texts, labels = texts.to(device), labels.to(device)
                val_outputs = model(texts)
                v_loss = criterion(val_outputs, labels)
                val_loss += v_loss.item()

        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)
        
        print(f"Epoch {epoch+1} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f} | Time: {time.time()-start_time:.1f}s")
        
        # --- KIỂM TRA PATIENCE ---
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), 'best_model.pt')
            print("✨ Val Loss improved! Saved new best model.")
            counter = 0 # Reset biến đếm nếu có cải thiện
        else:
            counter += 1
            print(f"⚠️ No improvement. Patience counter: {counter}/{patience}")
            if counter >= patience:
                early_stop = True

# RNNLSTM model

In [17]:
class RNNLSTM(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, device = 'cuda'):
        super(RNNLSTM, self).__init__()
        self.device = device
        self.hidden_dim = hidden_dim
        self.embedded = nn.Embedding(vocab_size, embedding_dim)

        combined_dim = embedding_dim + hidden_dim
        
        self.W_f = nn.Linear(combined_dim, hidden_dim) # Forget Gate (Cổng quên)
        self.W_i = nn.Linear(combined_dim, hidden_dim) # Input Gate (Cổng vào)
        self.W_c = nn.Linear(combined_dim, hidden_dim) # Cell Candidate (Ứng viên bộ nhớ mới)
        self.W_o = nn.Linear(combined_dim, hidden_dim) # Output Gate (Cổng ra)
        

        self.head_classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim // 2, output_dim)
        )

        # Thêm vào cuối __init__
        nn.init.constant_(self.W_f.bias, 1.0)
        
    def forward(self, x):
        batch_size = x.size(0)
        seq_length = x.size(1)
        
        embedded = self.embedded(x)
        
        # LSTM cần khởi tạo HAI trạng thái:
        # 1. Hidden State (Trạng thái ẩn / Bộ nhớ ngắn hạn)
        h_t = torch.zeros(batch_size, self.hidden_dim, device=self.device)
        # 2. Cell State (Trạng thái tế bào / Bộ nhớ dài hạn)
        C_t = torch.zeros(batch_size, self.hidden_dim, device=self.device)

        for t in range(seq_length):
            x_t = embedded[:, t, :]
            
            # Ghép nối x_t và h_t để đưa vào các cổng cùng lúc
            # Kích thước combined: [batch_size, embedding_dim + hidden_dim]
            combined = torch.cat((x_t, h_t), dim=1)
            
            # 1. Cổng Quên (Forget Gate): Quyết định vứt bỏ gì từ C_t cũ
            f_t = torch.sigmoid(self.W_f(combined))
            
            # 2. Cổng Vào (Input Gate): Quyết định cập nhật thông tin gì mới
            i_t = torch.sigmoid(self.W_i(combined))
            
            # 3. Ứng viên mới: Tạo ra các giá trị mới có thể thêm vào C_t
            c_tilde = torch.tanh(self.W_c(combined))
            
            # 4. Cập nhật Cell State (C_t): Kết hợp trí nhớ cũ và thông tin mới
            C_t = C_t * f_t + i_t * c_tilde

            # 5. Cổng Ra (Output Gate): Quyết định phần nào của C_t sẽ được xuất ra
            o_t = torch.sigmoid(self.W_o(combined))

            # 6. Cập nhật Hidden State (h_t): Tạo bộ nhớ ngắn hạn cho bước tiếp theo
            h_t = o_t * torch.tanh(C_t)

        # Đưa Hidden state cuối cùng qua Classifier Head
        out = self.head_classifier(h_t)
        
        return out
            

In [18]:
class PytorchRNNLSTM(nn.Module):
# ADDED: pretrained_embeddings parameter
    def __init__(self, vocab_size, embedding_dim, hidden_dim, output_dim, pretrained_embeddings=None, device='cuda'):
        super(PytorchRNNLSTM, self).__init__()
        self.device = device

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        
        # --- NEW: Load Pretrained Weights ---
        if pretrained_embeddings is not None:
            # Nạp trọng số GloVe
            self.embedding.weight = nn.Parameter(pretrained_embeddings)
            # Thường thì nên để True để model tinh chỉnh thêm (Fine-tune) theo data Toxic
            self.embedding.weight.requires_grad = False
            # OPTIONAL: Freeze the embeddings so they don't get messed up during early training
            # self.embedding.weight.requires_grad = False 
            print("Loaded pretrained embeddings successfully!")

        self.lstm = nn.LSTM(input_size=embedding_dim, 
                            hidden_size=hidden_dim, 
                            batch_first=True,
                            bidirectional=True) # I also sneaked in Bi-LSTM here for better accuracy!

        # Note: If using bidirectional=True, the hidden_dim going into the Linear layer doubles
        lstm_output_dim = hidden_dim * 2 

        self.head_classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(lstm_output_dim, lstm_output_dim // 2),
            nn.LayerNorm(lstm_output_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(lstm_output_dim // 2, output_dim)
        )
    def forward(self, x):
        # x shape: [batch_size, seq_length]
        
        embedded = self.embedding(x)
        # embedded shape: [batch_size, seq_length, embedding_dim]
        
        # Đưa dữ liệu qua LSTM
        # Khác với tự code, nn.LSTM tự động chạy hết toàn bộ câu trong một lần gọi
        lstm_out, (h_n, c_n) = self.lstm(embedded)
        
        # Lấy hidden state của từ CUỐI CÙNG trong câu
        # lstm_out chứa hidden state của TẤT CẢ các từ. 
        # Cú pháp [:, -1, :] nghĩa là: Lấy tất cả batch, ở bước thời gian cuối cùng (-1), lấy toàn bộ hidden_dim
        final_hidden_state = lstm_out[:, -1, :]
        
        # Đưa qua Classifier Head để dự đoán 6 nhãn
        out = self.head_classifier(final_hidden_state)
        
        return out

# Transformer

In [19]:
import torch
import torch.nn as nn
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len = 50000):
        super(PositionalEncoding, self).__init__()

        # Initialize a matrix positional encoding [max_len x d_model]
        pe = torch.zeros(max_len, d_model)

        # We calculate the 1 / (10000^(2 * i / d_model))
        divterm = torch.exp(torch.arange(0, d_model, 2) * (- math.log(10000) / d_model))

        # Initialize a array size max_len x 1.
        position = torch.arange(0, max_len, dtype = torch.float).unsqueeze(1)

        pe[:, 0 : : 2] = torch.sin(position * divterm)
        pe[:, 1 : : 2] = torch.cos(position * divterm)

        # Convert to [1 x max_len x d_model]
        pe = pe.unsqueeze(0)

        self.register_buffer('pe', pe)
    
    def forward(self, x):
        " x: Tensor, shape ``[batch_size, seq_len, embedding_dim]`` "
        
        x = x + self.pe[:, : x.size(1) : ]
        return x



class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads=4, hidden_dim=256):
        super(MultiHeadAttention, self).__init__()
        assert hidden_dim % num_heads == 0, "hidden_dim phải chia hết cho num_heads"
        
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads

        # Khởi tạo các ma trận trọng số (bias=False)
        self.Wv = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.Wk = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.Wq = nn.Linear(hidden_dim, hidden_dim, bias=False)

        # Lớp Linear cuối cùng sau khi gộp các đầu
        self.W_out = nn.Linear(hidden_dim, hidden_dim)

    def split_into_head(self, x):
        batch_size, seq_length, hidden_dim = x.shape
        # Tách chiều và transpose để đưa num_heads lên trước
        x = x.view(batch_size, seq_length, self.num_heads, self.head_dim)
        return x.transpose(1, 2) 
        # Output shape: [batch, num_heads, seq_length, head_dim]

    def forward(self, x):
        batch_size, seq_length, _ = x.shape
        
        # 1. Chiếu qua các lớp Linear (Projections)
        q = self.Wq(x)
        k = self.Wk(x)
        v = self.Wv(x)

        # 2. Tách đầu xử lý song song (Sửa lỗi gọi hàm: dùng self.split_into_head)
        q = self.split_into_head(q)
        k = self.split_into_head(k)
        v = self.split_into_head(v)

        # 3. Scaled Dot-Product Attention
        # Dùng math.sqrt hoặc self.head_dim**0.5
        attention_scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # 4. Tính trọng số (Attention Weights)
        attention_weights = torch.softmax(attention_scores, dim=-1)

        # 5. Tổng hợp thông tin từ Value (V)
        # Sửa lỗi tên biến: attenweights -> attention_weights
        attention_output = torch.matmul(attention_weights, v)

        # 6. Gộp các đầu lại (Concatenation)
        # Đưa về lại [batch, seq, heads, head_dim] sau đó nén thành [batch, seq, hidden_dim]
        attention_output = attention_output.transpose(1, 2).contiguous()
        attention_output = attention_output.view(batch_size, seq_length, self.hidden_dim)

        # 7. Final Linear layer
        return self.W_out(attention_output)


class TransformerBlock(nn.Module):
    def __init__(self, hidden_dim = 256, num_heads = 4, dropout = 0.1):
        super(TransformerBlock, self).__init__()

        self.attention = MultiHeadAttention(num_heads, hidden_dim)

        self.feed_forward = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.ReLU(),
            nn.Linear(hidden_dim * 2, hidden_dim)
        )

        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # --- BƯỚC 1: ATTENTION ---
        # Residual Connection: x + Attention(x)
        attn_out = self.attention(x)
        x = self.norm1(x + self.dropout(attn_out))
        
        # --- BƯỚC 2: FEED FORWARD ---
        # Residual Connection: x + FFN(x)
        ff_out = self.feed_forward(x)
        x = self.norm2(x + self.dropout(ff_out))

        return x 

class TransformerModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, output_dim, num_heads = 4, dropout = 0.1, num_layers = 2):
        super(TransformerModel, self).__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.pos_encoding = PositionalEncoding(embedding_dim)

        self.transformer_blocks = nn.ModuleList([
            TransformerBlock(embedding_dim, num_heads, dropout) 
            for _ in range(num_layers)
        ])
        
        # 4. Lớp Classifier cuối cùng cho 6 nhãn (Toxic, Severe Toxic, Obscene, Threat, Insult, Identity Hate)
        self.classifier_head = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(embedding_dim, embedding_dim // 2),
            nn.LayerNorm(embedding_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(embedding_dim // 2, output_dim)
        )

    def forward(self, x):
        x = self.embedding(x)
        x = self.pos_encoding(x)

        for block in self.transformer_blocks:
            x = block(x)
            
        # B3: Global Average Pooling (Lấy trung bình toàn bộ câu)
        # Tại sao? Vì ta cần 1 vector đại diện cho cả câu để phân loại
        x = x.mean(dim=1) # -> [batch_size, embedding_dim]
        
    
        logits = self.classifier_head(x)    
        return logits
        

In [20]:
import torch
import torch.nn as nn
import math

class PytorchPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super(PytorchPositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # Shape: [1, max_len, d_model]
        
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x shape: [batch_size, seq_len, d_model]
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class PytorchTransformerModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim, output_dim, num_heads=8, 
                 num_layers=4, dim_feedforward=1024, dropout=0.3):
        super(PytorchTransformerModel, self).__init__()

        # 1. Embedding & Positional Encoding
        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.pos_encoding = PytorchPositionalEncoding(embedding_dim, dropout=dropout)

        # 2. Built-in Transformer Encoder Layer
        # batch_first=True makes it compatible with [batch, seq, feature] layout
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embedding_dim,
            nhead=num_heads,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True 
        )

        # 3. Stack layers using TransformerEncoder
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # 4. Classifier Head
        self.classifier_head = nn.Sequential(
            nn.Linear(embedding_dim, embedding_dim // 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(embedding_dim // 2, output_dim)
        )

    def forward(self, x):
        # x shape: [batch_size, seq_len]
        
        # Pass through embedding and pos encoding
        x = self.embedding(x) # [batch, seq, emb]
        x = self.pos_encoding(x)

        # Pass through the full Transformer stack
        # By setting batch_first=True in the layer, we don't need to transpose x
        x = self.transformer_encoder(x) # [batch, seq, emb]
            
        # Global Average Pooling (representing the whole sentence)
        x = x.mean(dim=1) # [batch, emb]
        
        logits = self.classifier_head(x)    
        return logits

# Pretrained Glove

In [21]:
use_Glove = False

if use_Glove:        
    !wget http://nlp.stanford.edu/data/glove.6B.zip
    !unzip glove.6B.zip

In [22]:
import numpy as np

def load_glove_embeddings(vocab, glove_path, embedding_dim=100):
    vocab_size = len(vocab)
    # Khởi tạo ma trận ngẫu nhiên (hoặc toàn 0)
    embedding_matrix = np.random.uniform(-0.01, 0.01, (vocab_size, embedding_dim))
    
    print(f"Loading GloVe from {glove_path}...")
    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            if word in vocab:
                idx = vocab[word]
                vector = np.asarray(values[1:], dtype='float32')
                embedding_matrix[idx] = vector
    
    return torch.from_numpy(embedding_matrix).float()

# Giả sử bạn dùng bản GloVe 100d
# pretrained_weights = load_glove_embeddings(vocab, 'glove.6B.100d.txt', 100)

In [23]:
import time

if train_on_RNNLSTM:

    if use_Glove:
        # 1. Khởi tạo
        emb_dim = 100 
        pretrained_weights = load_glove_embeddings(vocab, 'glove.6B.100d.txt', emb_dim)

    # 1. Khởi tạo với pretrained_weights
    model = PytorchRNNLSTM(vocab_size=len(vocab), 
                            embedding_dim=128, 
                            hidden_dim=256, 
                            output_dim=6).to(device)

    criterion = nn.BCEWithLogitsLoss() 
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

    num_epochs = 60 # Bạn có thể để số epoch lớn hơn vì đã có Early Stopping
    best_val_loss = float('inf')
    
    # --- CẤU HÌNH PATIENCE ---
    patience = 7      # Nếu sau 3 Epoch mà Val Loss không giảm thì dừng
    counter = 0       # Biến đếm số lần không cải thiện
    early_stop = False

    print("Starting the training process with Early Stopping...")

    for epoch in range(num_epochs):
        if early_stop:
            print("🛑 Early stopping triggered. Training finished!")
            break

        start_time = time.time()
        
        # --- TRAINING PHASE ---
        model.train()
        train_loss = 0.0
        for texts, labels in train_loader:
            texts, labels = texts.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(texts)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()

        # --- VALIDATION PHASE ---
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for texts, labels in val_loader:
                texts, labels = texts.to(device), labels.to(device)
                val_outputs = model(texts)
                v_loss = criterion(val_outputs, labels)
                val_loss += v_loss.item()

        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)
        
        print(f"Epoch {epoch+1} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f} | Time: {time.time()-start_time:.1f}s")
        
        # --- KIỂM TRA PATIENCE ---
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), 'best_model.pt')
            print("✨ Val Loss improved! Saved new best model.")
            counter = 0 # Reset biến đếm nếu có cải thiện
        else:
            counter += 1
            print(f"⚠️ No improvement. Patience counter: {counter}/{patience}")
            if counter >= patience:
                early_stop = True

In [24]:
import time

if train_on_Transformer:

    model = PytorchTransformerModel(vocab_size=len(vocab), 
                            embedding_dim=256, 
                            output_dim=6).to(device)

    criterion = nn.BCEWithLogitsLoss() 
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0001)

    num_epochs = 60 # Bạn có thể để số epoch lớn hơn vì đã có Early Stopping
    best_val_loss = float('inf')
    
    # --- CẤU HÌNH PATIENCE ---
    patience = 7      # Nếu sau 3 Epoch mà Val Loss không giảm thì dừng
    counter = 0       # Biến đếm số lần không cải thiện
    early_stop = False

    print("Starting the training process with Early Stopping...")

    for epoch in range(num_epochs):
        if early_stop:
            print("🛑 Early stopping triggered. Training finished!")
            break

        start_time = time.time()
        
        # --- TRAINING PHASE ---
        model.train()
        train_loss = 0.0
        for texts, labels in train_loader:
            texts, labels = texts.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(texts)
            loss = criterion(outputs, labels)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss += loss.item()

        # --- VALIDATION PHASE ---
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for texts, labels in val_loader:
                texts, labels = texts.to(device), labels.to(device)
                val_outputs = model(texts)
                v_loss = criterion(val_outputs, labels)
                val_loss += v_loss.item()

        avg_train = train_loss / len(train_loader)
        avg_val = val_loss / len(val_loader)
        
        print(f"Epoch {epoch+1} | Train Loss: {avg_train:.4f} | Val Loss: {avg_val:.4f} | Time: {time.time()-start_time:.1f}s")
        
        # --- KIỂM TRA PATIENCE ---
        if avg_val < best_val_loss:
            best_val_loss = avg_val
            torch.save(model.state_dict(), 'best_model.pt')
            print("✨ Val Loss improved! Saved new best model.")
            counter = 0 # Reset biến đếm nếu có cải thiện
        else:
            counter += 1
            print(f"⚠️ No improvement. Patience counter: {counter}/{patience}")
            if counter >= patience:
                early_stop = True

Starting the training process with Early Stopping...
Epoch 1 | Train Loss: 0.0917 | Val Loss: 0.0716 | Time: 180.9s
✨ Val Loss improved! Saved new best model.
Epoch 2 | Train Loss: 0.0658 | Val Loss: 0.0646 | Time: 186.9s
✨ Val Loss improved! Saved new best model.
Epoch 3 | Train Loss: 0.0608 | Val Loss: 0.0556 | Time: 186.8s
✨ Val Loss improved! Saved new best model.
Epoch 4 | Train Loss: 0.0569 | Val Loss: 0.0522 | Time: 186.9s
✨ Val Loss improved! Saved new best model.
Epoch 5 | Train Loss: 0.0540 | Val Loss: 0.0537 | Time: 186.5s
⚠️ No improvement. Patience counter: 1/7
Epoch 6 | Train Loss: 0.0518 | Val Loss: 0.0522 | Time: 186.8s
⚠️ No improvement. Patience counter: 2/7
Epoch 7 | Train Loss: 0.0502 | Val Loss: 0.0526 | Time: 186.8s
⚠️ No improvement. Patience counter: 3/7
Epoch 8 | Train Loss: 0.0489 | Val Loss: 0.0494 | Time: 186.7s
✨ Val Loss improved! Saved new best model.
Epoch 9 | Train Loss: 0.0477 | Val Loss: 0.0512 | Time: 186.8s
⚠️ No improvement. Patience counter: 1/7
E

# Creat test loader

In [25]:
import re
def simple_clean(text):
    text = text.lower()
    text = re.sub(r'[\n\t\r]', ' ', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text

df_test['comment_text'] = df_test['comment_text'].astype(str).apply(simple_clean)

df_test


,id,comment_text
0,00001cee341fdb12,yo bitch ja rule is more succesful then youll ...
1,0000247867823ef7,from rfc the title is fine as it is imo
2,00013b17ad220c46,sources zawe ashton on lapland
3,00017563c3f7919a,if you have a look back at the source the info...
4,00017695ad8997eb,i dont anonymously edit articles at all
...,...,...
153159,fffcd0960ee309b5,i totally agree this stuff is nothing but t...
153160,fffd7a9a6eb32c16,throw from out field to home plate does i...
153161,fffda9e8d6fafa9e,okinotorishima categories i see your ...
153162,fffe8f1340a79fc2,one of the founding nations of the eu ge...


In [26]:
# 1. Create the Test Dataset
# We pass dummy labels (zeros) because we don't know the real ones yet
dummy_labels = np.zeros((len(df_test), 6)) 

test_dataset = ToxicDataset(
    df_test['comment_text'].values, 
    dummy_labels, 
    word_to_idx=vocab, 
    max_len=100
)

# 2. Create the Test Loader (Shuffle MUST be False here)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

In [27]:
# 1. Load the best weights
model.load_state_dict(torch.load('best_model.pt'))
model.eval()

test_predictions = []

print("Generating predictions for test set...")
with torch.no_grad():
    for texts, _ in test_loader: # Assuming you made a test_loader
        texts = texts.to(device)
        
        # Get raw outputs
        outputs = model(texts)
        
        # Convert logits to probabilities (0.0 to 1.0)
        probs = torch.sigmoid(outputs)
        
        test_predictions.append(probs.cpu().numpy())

# 2. Flatten the list of batches into one big array
test_predictions = np.vstack(test_predictions)

# 3. Create the Submission DataFrame
submission = pd.DataFrame(test_predictions, columns=['toxic', 'severe_toxic', 'obscene', 'threat', 'insult', 'identity_hate'])
submission.insert(0, 'id', df_test['id'].values) # Add the ID column back

# 4. Save to CSV
submission.to_csv('submission.csv', index=False)

Generating predictions for test set...
